# Cryoscope Experiment

-----------------------------------------

Written by Dharun Venkateswaran, QES Intern @ Keysight Technologies, Edinburgh.

Initial experiments performed on the QPU at NUS (National University of Singapore).

Grateful appreciation given to Francesco Tafuri, Bruno Aznar Martinez, and Paul Tan for their fruitful suggestions, guidance, and helpful discussions. 

Note: Keysight employees can visit the Confluence page [https://confluence.it.keysight.com/display/NGSup/Cryoscope] for the full documentation and detailed explanation of the notebook. They can also visit the BitBucket page [https://bitbucket.it.keysight.com/projects/QES/repos/qcs-api-prog-examples/browse/Dharuns%20Sandbox/Cryoscope] for prototype code and Paul's code is found at [https://bitbucket.it.keysight.com/projects/QES/repos/qcs-api-prog-examples/browse/Dharuns%20Sandbox/Cryoscope/NUS%20Code/cryoscope_raw.ipynb].

--------------------------------------------

High-fidelity two-qubit gates are desired to deploy reliable quantum algorithms for quantum computing. For superconducting qubits, microwave pulses are used for qubit control, but non-ideal electronic components (e.g. impurities) in the signal chain result in dynamical distortions (DDs) for the microwave pulse, hindering performance. Digital PreDistortion (DPD) algorithms can correct this _if_ the distortions are known. 

Cryoscope [1] (a portmanteu of Cryogenic and Oscilloscope) is a State-of-the-Art (SoA) technique which aims to characterise the DDs in the flux control line by reconstructing the qubit's voltage-to-flux response, effectively treating it like an oscilloscope. As an application, one can then apply IIR (Infinite Impulse Response) and FIR (Finite Impulse Response) filters in real-time which is the DPD algorithm that will correct the DDs, hence achieving improved qubit control reuslting in higher fidelity two-qubit gates as desired. 

This notebook will perform the Cryoscope experiment in its entirety, including the DPD steps. The workflow consists of the following steps:

1. Run the Ramsey-style experiment and collect the I/Q data.
2. Demodulate the highest-frequency part of the data using a FFT.
3. Find the accumulated phase of the qubit, unwrap it and apply a smoothing filter.
4. Determine the detuning frequency of the qubit and use eq. (1) from [1] to reconstruct the voltage-to-flux response.
5. Run the DPD algorithm consisting of an IIR and FIR filter to correct the response. 
6. Re-run steps 1 - 4, hoping for a corrected flux response.



[1]: M. A. Rol et al., Time-domain characterization and correction of on-chip
distortion of control pulses in a quantum processor, Applied Physics Letters (2020).

----------------------------------

## Beginning Steps

------------------------------

There are a few initial steps that need to be completed before the experiment can be ran. More information about each of them is given in their subsection.

They involve:
1. Initialising the HW and importing required libraries,
2. Importing the values from the calibration set and assigning channels via the channel mapper, 
3. Training the classifier and checking for high-fidelity initial readouts,
4. Extracting the required physical HW parameters.


### Initialising Code

----------------------------------------------

The following imports the required libraries, calibration set and channel mapper, and initialises the QPU to be used. Check that the backend systems check reads as 'True' at the end before proceeding.


In [ ]:
import keysight.qcs as qcs
import numpy as np
import matplotlib.pyplot as plt
import json
import re
import time
import copy
from keysight.qcs.utils import load 
from scipy.fft import fft, ifft, fftfreq
from scipy.signal import savgol_filter, lfilter
from scipy.constants import h
from scipy.optimize import curve_fit
from scipy.signal import find_peaks
from keysight.qcs.experiments import GATES, Experiment 
from keysight.qcs.analysis import Exponential, IQuadrature, DecayingSinusoid
from matplotlib.colors import BoundaryNorm
from matplotlib.ticker import MaxNLocator
from pathlib import Path

print(qcs.__version__)

c:\Python\Keysight\TSM\lib\site-packages\keysight\qcs\interfaces\qasm_interface\qasm_converter.py:31: DeprecationWarning: Using Qiskit with Python 3.9 is deprecated as of the 2.1.0 release. Support for running Qiskit with Python 3.9 will be removed in the 2.3.0 release, which coincides with when Python 3.9 goes end of life.
  from qiskit.qasm import libs as qasm_lib


2.6.0a3


In [ ]:
# Your JSON data as a string
data_str = '''
{
  "qubit_offsets": {
    "0": "0",
    "1": "0.2",
    "2": "0",
    "3": "0.18",
    "4": "0",
    "5": "-0.22",
    "6": "0",
    "7": "0.15",
    "8": "0",
    "9": "-0.18",
    "10": "0",
    "11": "0.1",
    "12": "0",
    "13": "0.109",
    "14": "0",
    "15": "0",
    "16": "0",
    "17": "0",
    "18": "0.15",
    "19": "0"
  },
  "coupler_offsets": {
    "TC 0-3": "0.078",
    "TC 2-3": "0.142",
    "TC 2-7": "0.087",
    "TC 7-12": "0.149",
    "TC 12-13": "0.06",
    "TC 13-17": "0.108",
    "TC 17-18": "0.057",
    "TC 18-19": "0.107",
    "TC 15-19": "0.0846",
    "TC 15-16": "0.135",
    "TC 11-16": "0.08",
    "TC 6-11": "0.096",
    "TC 5-6": "0.15",
    "TC 4-5": "0.161",
    "TC 1-4": "-0.12",
    "TC 3-4": "0.154",
    "TC 4-9": "0.09",
    "TC 0-1": "0.137",
    "TC 8-9": "0.117",
    "TC 3-8": "0.076",
    "TC 7-8": "0.0575",
    "TC 8-13": "0.0858",
    "TC 14-15": "0.116",
    "TC 9-14": "0.088",
    "TC 13-14": "0.059",
    "TC 14-18": "0.1355",
    "TC 10-11": "0.145",
    "TC 5-10": "0.092",
    "TC 9-10": "0.137",
    "TC 10-15": "0.084"
  }
}
'''

# Load JSON
data_dict = json.loads(data_str)

# --- Parse qubit_offsets (int keys, float values, skip zeros) ---
qubit_offsets = {
    int(k): float(v)
    for k, v in data_dict["qubit_offsets"].items()
    if float(v) != 0.0
}

# --- Parse coupler_offsets ("TC a-b": "offset") ---
coupler_offsets = {}
for key, val in data_dict["coupler_offsets"].items():
    # Extract numbers from key like "TC 0-3"
    match = re.search(r"(\d+)-(\d+)", key)
    if match:
        q1, q2 = int(match.group(1)), int(match.group(2))
        coupler_offsets[(q1, q2)] = float(val)

# Combine results
parsed_data = {
    "qubit_offsets": qubit_offsets,
    "coupler_offsets": coupler_offsets
}


In [ ]:
def generate_program_and_update_mapper(json_data, channel_mapper, qubits) -> tuple[qcs.Program, qcs.ChannelMapper]:
    coupler_bias = json_data["coupler_offsets"]
    qubit_bias = json_data["qubit_offsets"]
    MIN_PULSE_LENGTH = 20e-9

    volt = 2.7

    # Define hardware topology
    single_flux_qubit = [1, 3, 5, 7, 9, 11, 13, 18]
    connectivity = [
        (0, 1),
        (0, 3), 
        (1, 4), 
        (2, 3), 
        (2, 7), 
        (3, 4), 
        (3, 8), 
        (4, 5), 
        (4, 9), 
        (5, 6), 
        (5, 10), 
        (6, 11), 
        (7, 8), 
        (7, 12), 
        (8, 9), 
        (8, 13), 
        (9, 10), 
        (9, 14), 
        (10, 11),
        (10, 15), 
        (11, 16), 
        (12, 13), 
        (13, 14), 
        (13, 17), 
        (14, 15), 
        (14, 18), 
        (15, 16), 
        (15, 19), 
        (17, 18), 
        (18, 19)
    ]

    # --- Unpack channels from mapper ---
    xy_awg_chan, ro_awg_chan, ro_dig_chan, qb_flux_chan, tc_flux_chan, _, _ = channel_mapper.channels

    # --- Build mapping dictionaries ---
    coupler_map: dict[tuple[int, int], qcs.Channels] = {
        (a, b): channel for (a, b), channel in zip(connectivity, tc_flux_chan)
    }
    qubit_map: dict[int, qcs.Channels] = {
        a: channel for a, channel in zip(single_flux_qubit, qb_flux_chan)
    }

    # --- Create program ---
    program = qcs.Program()

    # --- Add coupler biases (and update mapper offsets) ---
    added_couplers = set()
    for coupler, bias in coupler_bias.items():
        if (coupler[0] in qubits or coupler[1] in qubits) and coupler not in added_couplers:
            # Add DC waveform to program
            program.add_waveform(
                pulse=qcs.DCWaveform(
                    duration=MIN_PULSE_LENGTH,
                    amplitude=0,
                    envelope=qcs.ConstantEnvelope()
                ),
                channels=coupler_map[coupler]
            )

            # Update channel mapper offset
            channel_mapper.get_physical_channels(coupler_map[coupler])[0].settings.offset.value = bias * volt

            added_couplers.add(coupler)

    # --- Add single-qubit biases (and update mapper offsets) ---
    for qubit, bias in qubit_bias.items():
        if qubit in qubits:
            # Add DC waveform to program
            program.add_waveform(
                pulse=qcs.DCWaveform(
                    duration=MIN_PULSE_LENGTH,
                    amplitude=0,
                    envelope=qcs.ConstantEnvelope()
                ),
                channels=qubit_map[qubit]
            )

            # Update channel mapper offset
            channel_mapper.get_physical_channels(qubit_map[qubit])[0].settings.offset.value = bias * volt
    program.add_delay(50e-9,coupler_map[list(added_couplers)[0]],new_layer=True)
    program.add_delay(50e-9,coupler_map[list(added_couplers)[-1]],new_layer=True)

    # Return both
    return program, channel_mapper


In [ ]:
channel_mapper = qcs.load("chan_map.qcs")
calibration_set = qcs.load("qubit0n1_calibrated.qcs")
# you can export and import these changed/updated values
"""calset.import_values("calset_values_v2.json")
calset.export_values("calset_values_v3.json")
# you can save entire object and load
qcs.save(cal_set, "cal_set_v4.qcs")
cal_Set_v4 = qcs.load("cal_set_v4.qcs")"""

In [ ]:
init_program, channel_mapper = generate_program_and_update_mapper(parsed_data, channel_mapper, [0,1])

backend = qcs.HclBackend(channel_mapper, fpga_postprocessing = True, init_time=300E-6, suppress_rounding_warnings=True)
backend.is_system_ready()

### Channel Assignments and Wave Pulses

----------------------------------------------

This section utilises the channel mapper and calibration set to assign the virtual channels to the physical ones and finds the relevant parameter values for the waveforms that will later be used.


In [ ]:
n_qubits = 2 # then select [1] from each channel
qubits = qcs.Qudits(range(n_qubits), 'Qubits') # and do [0,3] to select 0 and 3, for example
xy_awgs = qcs.Channels(range(n_qubits), "xy_channels", absolute_phase=False)
readout_awgs = qcs.Channels(range(n_qubits), "readout_channels", absolute_phase=True)
qubit_flux_awgs = qcs.Channels(range(n_qubits), "fastflux_qubit_channels", absolute_phase=True)
dig_channel = qcs.Channels(range(n_qubits), "readout_acquisition", absolute_phase=True)

# take parameters directly from calibration set; only worry is what happens when they update
rot_angle = np.pi / 2 
# choose qubit 1 bc we send a pulse down the flux line therefore need a flux-tunable qubit
pulse_duration = calibration_set.variables.single_qubit_durations.value[1] # index choosing qubit 1?
pulse_amplitude = calibration_set.variables.x180_pulse_amplitudes.value[1] # (rot_angle / np.pi) * 1/2
pulse_freq = calibration_set.variables.xy_pulse_frequencies.value[1]

readout_duration = calibration_set.variables.readout_durations.value[1] # same as acquisition (usually)
readout_amp = calibration_set.variables.readout_pulse_amplitudes.value[1]
readout_freq = calibration_set.variables.readout_frequencies.value[1]

# pulse level
pi2_ypulse = qcs.RFWaveform(
    duration = 1/2*pulse_duration,
    envelope = qcs.GaussianEnvelope(),
    amplitude = pulse_amplitude,
    rf_frequency = pulse_freq,
    instantaneous_phase = np.pi /2
)
pi2_xpulse = qcs.RFWaveform(
    duration = 1/2*pulse_duration,
    envelope = qcs.GaussianEnvelope(),
    amplitude = pulse_amplitude,
    rf_frequency = pulse_freq,
    instantaneous_phase = 0
)
readout_xpulse = qcs.RFWaveform( 
    duration = readout_duration,
    envelope = qcs.GaussianEnvelope(),
    amplitude = readout_amp,
    rf_frequency = readout_freq
)
readout_ypulse = qcs.RFWaveform( 
    duration = readout_duration,
    envelope = qcs.GaussianEnvelope(),
    amplitude = readout_amp,
    rf_frequency = readout_freq,
    instantaneous_phase = np.pi / 2
)

------------------------------

### Classifier Training
------------------------------------------------------------------------------------------------

Throughout this section, whenever fidelities deviate away by more than 5% away from the ideal, a ValueError is thrown. This part HAS to be run if using Method 1 of performing the experiment as it relies on the classifier reference. 

Each pulse used in the Ramsey-style experiment should be calibrated first. This can be done by running simple experiments and checking that the expected state measurements from the classifier data (no. of 0s and 1s) are what they should be. We are looking for high-fidelities of >95% for *both* the ground and excited state.

If it _isn't_ then that lets us know something is fundamentally wrong with the HW which should be troubleshooted first. The other possible error could be due to the parameters of the wavepulses which are taken from the calibration set that have to be recalibrated.

First, we create the classifier reference to be checked against by running experiments that we know should give all 0s and all 1s. Then we run experiments knowing they should result in the ground and excited state and check how well the HW measures their population level. Here is where we are aiming for >95% fidelity first.

In [ ]:
gnd_sequence = copy.deepcopy(init_program)
gnd_sequence.n_shots(10000)
gnd_sequence.add_waveform(readout_xpulse, readout_awgs[1], new_layer = True)
gnd_sequence.add_acquisition(integration_filter = readout_xpulse, channels = dig_channel[1], new_layer = False, pre_delay = 10e-9)
gnd_sequence = qcs.LinkerPass(*calibration_set.linkers.values()).apply(gnd_sequence)

exc_sequence = copy.deepcopy(init_program)
exc_sequence.n_shots(10000)
exc_sequence.add_waveform([pi2_xpulse, pi2_xpulse], xy_awgs[1], new_layer = True)
exc_sequence.add_waveform(readout_xpulse, readout_awgs[1], new_layer = True)
exc_sequence.add_acquisition(integration_filter = readout_xpulse, channels = dig_channel[1], new_layer = False, pre_delay = 10e-9)
exc_sequence = qcs.LinkerPass(*calibration_set.linkers.values()).apply(exc_sequence)

res_gnd = backend.apply(gnd_sequence)
res_exc = backend.apply(exc_sequence)

iq_gnd = next(iter(res_gnd.results.get_iq(dig_channel[1]).values()))
iq_exc = next(iter(res_exc.results.get_iq(dig_channel[1]).values()))

classifier_ref = [
    np.average(iq_gnd), np.average(iq_exc)
]

In [ ]:
gnd_sequence = copy.deepcopy(init_program)
gnd_sequence.n_shots(1000)
gnd_sequence.add_waveform(readout_xpulse, readout_awgs[1], new_layer = True)
gnd_sequence.add_acquisition(integration_filter = readout_xpulse, channels = dig_channel[1], classifier = qcs.Classifier(classifier_ref), new_layer = False, pre_delay = 10e-9)
gnd_sequence = qcs.LinkerPass(*calibration_set.linkers.values()).apply(gnd_sequence)

exc_sequence = copy.deepcopy(init_program)
exc_sequence.n_shots(1000)
exc_sequence.add_waveform([pi2_xpulse, pi2_xpulse], xy_awgs[1], new_layer = True)
exc_sequence.add_waveform(readout_xpulse, readout_awgs[1], new_layer = True)
exc_sequence.add_acquisition(integration_filter = readout_xpulse, channels = dig_channel[1], classifier = qcs.Classifier(classifier_ref), new_layer = False, pre_delay = 10e-9)
exc_sequence = qcs.LinkerPass(*calibration_set.linkers.values()).apply(exc_sequence)

res_gnd = backend.apply(gnd_sequence)
res_exc = backend.apply(exc_sequence)

In [ ]:
gnd_state_fidelity = np.round((1 - np.average(next(iter(res_gnd.results.get_classified(dig_channel[1]).values())))) * 100, decimals=3)
exc_state_fidelity = np.round(np.average(next(iter(res_exc.results.get_classified(dig_channel[1]).values()))) * 100, decimals=3)

print(f"|0>: {gnd_state_fidelity}%")
print(f"|1>: {exc_state_fidelity}%")

if gnd_state_fidelity < 95 or exc_state_fidelity < 95:
    raise ValueError("The population measurements are not accurate enough. Please troubleshoot the HW and try again.")

-------------------------------------------------------------------------

Now we are ready to check the result of the pulses that will be used in the Ramsey experiment. 
We expect that for both the \pi/2-pulses along the X and Y axis, one should end up with the qubit in an equal superposition of the 0 and 1 state (the difference between the two is an immeasurable phase). Hence, we expect a result close to 50%.

For the BB AWG pulse, there should be no measurable change to the qubit's state hence we expect it to be in the 0 state and expect a result as close to 100% as possible. 

In [ ]:
xpulse_sequence = copy.deepcopy(init_program)
xpulse_sequence.n_shots(5000)
xpulse_sequence.add_waveform(pi2_xpulse, xy_awgs[1], new_layer = True)
xpulse_sequence.add_waveform(readout_xpulse, readout_awgs[1], new_layer = True)
xpulse_sequence.add_acquisition(integration_filter = readout_xpulse, channels = dig_channel[1], classifier = qcs.Classifier(classifier_ref), new_layer = False, pre_delay = 20e-9)
xpulse_sequence = qcs.LinkerPass(*calibration_set.linkers.values()).apply(xpulse_sequence)

ypulse_sequence = copy.deepcopy(init_program)
ypulse_sequence.n_shots(5000)
ypulse_sequence.add_waveform(pi2_ypulse, xy_awgs[1], new_layer = True)
ypulse_sequence.add_waveform(readout_ypulse, readout_awgs[1], new_layer = True)
ypulse_sequence.add_acquisition(integration_filter = readout_ypulse, channels = dig_channel[1], classifier = qcs.Classifier(classifier_ref), new_layer = False, pre_delay = 20e-9)
ypulse_sequence = qcs.LinkerPass(*calibration_set.linkers.values()).apply(ypulse_sequence)


bb_pulse = qcs.RFWaveform(
        duration = 500e-9,
        envelope = qcs.ConstantEnvelope(),
        amplitude = 0.5,
        rf_frequency = 0
    )

bbpulse_sequence = copy.deepcopy(init_program)
bbpulse_sequence.n_shots(5000)
bbpulse_sequence.add_waveform(bb_pulse, qubit_flux_awgs[1], new_layer = True)
bbpulse_sequence.add_waveform(readout_xpulse, readout_awgs[1], new_layer = True)
bbpulse_sequence.add_acquisition(integration_filter = readout_xpulse, channels = dig_channel[1], classifier = qcs.Classifier(classifier_ref), new_layer = False, pre_delay = 20e-9)
bbpulse_sequence = qcs.LinkerPass(*calibration_set.linkers.values()).apply(bbpulse_sequence)

res_xpulse = backend.apply(xpulse_sequence)
res_ypulse = backend.apply(ypulse_sequence)
res_bbpulse = backend.apply(bbpulse_sequence)

In [ ]:
xpulse_pop = np.round((np.average(next(iter(res_xpulse.results.get_classified(dig_channel[1]).values())))) * 100, decimals=3)
ypulse_pop = np.round(np.average(next(iter(res_ypulse.results.get_classified(dig_channel[1]).values()))) * 100, decimals=3)
bbpulse_pop = np.round((1 - np.average(next(iter(res_bbpulse.results.get_classified(dig_channel[1]).values())))) * 100, decimals=3)

print(f"|0> + |1>: {xpulse_pop}%")
print(f"|0> + |1>: {ypulse_pop}%")
print(f"|0>: {bbpulse_pop}%")

xdeviation = abs(50.0 - xpulse_pop)
ydeviation = abs(50.0 - ypulse_pop)
bbdeviation = abs(100.0 - bbpulse_pop)

if xdeviation > 5.0 or ydeviation > 5.0 or bbdeviation > 5.0:
    raise ValueError("The population measurements are not accurate enough. Please troubleshoot the HW and try again.")


---------------------------------

### Flux Fitting: Qubit Spectroscopy 2D
--------------------------------------------
In order to invert eq. 1 from the Cryoscope paper and obtain the flux response, one needs to know the HW parameters Phi_0 (superconducting magnetic flux quantum), Phi_offset (the DC offset voltage), Ej (Josephson energy), and Ec (charging energy). Here we perform an experiment which will extract those parameters for us. 

In reality, as Ec is small, we get away with just finding the constant term required: \sqrt{8 x Ej x Ec}.

For this 2D Qubit Spectroscopy experiment, a long RF pulse is played to the qubit and the frequency of the pulse is swept. When the frequency of the pulse is resonant with a qubit transition frequency, there will be a stark change in the tranmission of the readout pulse. Simultaneously, we are also playing a BB AWG flux pulse to the qubit to change its frequency. Overall, this 2D scan of driving frequency and flux amplitude will allow us to find the qubit frequency as a function of the flux pulse amplitude--as desired by eq. 1 from the Cryoscope paper. From the results, we can then extract our desired parameters using standard known equations [2].

--- Ignore this part below. It will be relevant if there's a method to reliably find the second transition curve. ---

ValueErrors are thrown if the anharmonicity or the energy ratio deviate from the expected and desired values.
The penultimate cell contains a set of constants that can be used for the rest of the experiment if there is no time to re-run this experiment and re-calibrate their values. 


[2]: P. Krantz, et al., A quantum engineer's guide to superconducting qubits, Appl. Phys. Rev. 6, 021318, (2019)

In [ ]:
twoD_program = copy.deepcopy(init_program)

# config
pulse_length = 20e-6 # long RF pulse
nshots = 200

# sweeping parameters
flux_amplitude = qcs.Scalar("flux_amplitude", 0, dtype=float)
flux_amplitude_array = qcs.Array(
    name = "flux_amplitude_array",
    value = np.linspace(-0.3, 0.3, 100),
    dtype = float
)

driving_frequency = qcs.Scalar("driving_frequency", pulse_freq, dtype = float)
driving_frequency_array = qcs.Array(
    name = "driving_frequency_array",
    value = np.linspace(pulse_freq - 0.15e9, pulse_freq + 0.15e9, 100), 
    dtype = float
)

# driving pulse
twoD_program.add_waveform(
    pulse = qcs.RFWaveform(
        duration = pulse_length,
        envelope = qcs.GaussianEnvelope(),
        amplitude = pulse_amplitude,
        rf_frequency = driving_frequency,
    ),
    channels = xy_awgs[1],
    new_layer = True
)

# DC flattop
twoD_program.add_waveform(
    pulse = qcs.DCWaveform.create_dc_flattop(
        rise_duration = 20e-9,
        hold_duration = pulse_length - 40e-9,
        fall_duration = 20e-9,
        envelope = qcs.ConstantEnvelope(),
        amplitude = flux_amplitude
    ),
    channels = qubit_flux_awgs[0],
    pre_delay = 10e-9 # XY-Z channel timing sync
)

# readout
twoD_program.add_waveform(readout_xpulse, readout_awgs[1], new_layer = True)
twoD_program.add_acquisition(integration_filter = readout_xpulse, channels = dig_channel[1], new_layer = False, pre_delay = 53.33e-9) 


twoD_qubit = Experiment(backend = channel_mapper,
                        calibration_set = calibration_set,
                        targets = qubits,
                        program = twoD_program,
                        name = "2D Qubit Experiment"
                        )

twoD_qubit.configure_repetitions(
    n_shots = nshots,
    hw_sweep = 2,
    driving_frequency = driving_frequency_array,
    flux_amplitude = flux_amplitude_array
)

twoD_qubit.draw()

In [ ]:
twoD_qubit.execute(fpga_postprocessing = True)

In [ ]:
results = twoD_qubit.get_iq_array()

Now the results of the 2D Qubit Spectroscopy experiment are plotted to find the frequency as a function of the flux amplitude. Then the constansts that we are seeking can be extracted by fitting a curve to the result

In [ ]:
spec_res = results.real * 1e3
spec_res = spec_res.isel(channel = 0)
spec_res = spec_res.mean(dim = "shot")
spec_res = np.array(spec_res)

spec_x = driving_frequency_array.value / 1e9
spec_y = flux_amplitude_array.value

cpoint = 4.592 # freq for highest amplitude value (taken from next part)
freqs = []

for raw_array in spec_res:
    smooth = savgol_filter(raw_array, 11, 2)
    mask = np.logical_and(spec_x > (cpoint - 0.05), spec_x < (cpoint + 0.05))
    xmasked = spec_x[mask]
    ymasked = smooth[mask]

    cpoint = xmasked[np.argmax(ymasked)]
    freqs.append(cpoint)

freqs = np.array(freqs)

levels = MaxNLocator(nbins=100).tick_values(spec_res.min(), spec_res.max())
cmap = plt.get_cmap('inferno')
norm = BoundaryNorm(levels, ncolors=cmap.N, clip=True)
plt.imshow(spec_res, cmap, norm, extent=(spec_x[0], spec_x[-1], spec_y[-1], spec_y[0]), 
                aspect="auto", origin="lower", interpolation='none')
cbar = plt.colorbar()
cbar.set_label("Amplitude [ADC arb. units]")
plt.ylabel("Qubit Flux Pulse Amplitude [arb. units]")
plt.xlabel("Qubit Driving Frequency [GHz]")
plt.title(f"Q{1} Flux-Qubit Spectroscopy")

mask = spec_y < 0.1
freqs_smooth = savgol_filter(freqs[mask], 11, 2)
yfit = spec_y[mask]

plt.plot(freqs_smooth, -yfit, linestyle='dashed', label="Extracted Flux-Frequency Curve") # -yfit because y-axis is flipped in size 
plt.legend()
plt.title("Qubit Flux Spectroscopy")

plt.show()

In [ ]:
def flux_obj_func(x, *args):
    offset, wmax, fluxquanta = args

    if offset < 0:
        return wmax * np.sqrt(np.abs(np.cos((x - offset) / fluxquanta * np.pi)))

    elif offset > 0:
            return wmax * np.sqrt(np.abs(np.cos((x + offset) / fluxquanta * np.pi)))

# initial guess of the curve parameters
pguess = [
    -yfit[np.argmax(freqs_smooth)],
    freqs_smooth.max(),
    4
]

popt, pcov = curve_fit(flux_obj_func, yfit, freqs_smooth, p0=pguess)
fit = flux_obj_func(yfit, *popt)
offset, wmax, fluxquanta = popt

print(f"Offset voltage is {offset:.3f} (arb. units)")
print(f"Maximum frequency is {wmax:.3f} GHz")
print(f"Phi0 is {fluxquanta:.3f} (arb. units)")

plt.plot(-yfit, freqs_smooth)
plt.plot(-yfit, fit, linestyle='dashed', color="k")
plt.xlabel("Flux Pulse Amplitude [arb. units]")
plt.ylabel("Qubit Frequency [GHz]")
plt.grid()
plt.tight_layout()
plt.show()

In [ ]:
# if no time to re-run experiment, these values can be used
# last updated: 12/02/2026, 10:52 GMT UK time

offset = 0.221 # a.u
wmax = 4.609 # GHz units
fluxquanta = 2.236 # a. u


In [ ]:
# --- can ignore ---
# maybe if the second transition level can be nicely found to find alpha reliably

"""alpha = ((4.525 - wmax) * 1e9) * 2
print(f"The anharmonicity is {-alpha / 1e6:.2f} MHz")  # ensure 100 < alpha < 300 (MHz)
Ec = -1 * alpha # in natural units where h = 1, otherwise * h
print(f"The Charging Energy is {Ec}")

Ej = ( ((wmax * 1e9)) + Ec) ** 2 / (8 * Ec) # in natural units, otherwise ( (h * (wmax * 1e9)) + Ec) ** 2 / (8 * Ec)
print(f"The Josephson Energy is {Ej}")
energy_ratio = Ej / Ec

print(f"The energy ratio is {energy_ratio:.2f}") # ensure >= 50

if -alpha/1e6 < 100 or -alpha/1e6 > 300:
    raise ValueError("The anharmonicity is not in the desired range. Please troubleshoot the HW and try again.")

if energy_ratio < 50:
    raise ValueError("The energy ratio is not above 50. Please troubleshoot the HW and try again.")"""

----------------------------------

## Ramsey-Style Experiment for Cryoscope
----------------------------------------
Here, finally, the Ramsey-style experiment is performed to obtain the qubit's accumulated phase. 

Note: This first method of Cryoscope is not a 'neat' way of performing the experiment. The bulk of the distortion happens in the first 15ns and we would like to capture this. Unforunately, the QCS SW does not allow for wavepulses <13.333ns in duration nor can we sweep over envelopes. 
Hence, this is a clunky workaround for playing short pulses which has been developed instead. 
I am aware that the FPGA Team is currently developing a way of bypassing this problem but the scheduled update is due April 2026. 

The second method utilises the standard way of using Keysight's QCS SW but starts the BB AWG pulses at 15ns in duration, missing the key distortion information from the first 15ns.

Also, currently the only way to modify the BB AWG pulse with the DPD is using Method 1. Therefore, stick to it for now.



### Method 1

----------------------------------

This method uses an envelope which is looped over which varies the pulse duration. This workaround allows pulses to be played with <13.333ns duration.

Note that there will be MANY progress bars generated as it's not running just 'one' experiment because of the 'for loops'.
If one could sweep over envelopes in the qcs library then this problem could be overcome.

In [ ]:
def method1_cryoscope(pulse_duration, num_samples, tau_sweep, bb_awg, target_amplitude):
    """
    Executes the Ramsey-style experiment and is able to apply BB AWG pulses that are <13.333ns in duration. The results
    are given as the X and Y expectation values. The results are also saved as NumPy arrays so the data can be accessed
    and used without having to re-run the experiment. 

    Parameters
    -----------
    pulse_duration : float
        The longest BB AWG pulse to be played in the experiment. 
    num_samples : int (positive)
        Sets the number of BB AWG pulses of increasing duration that should be played. 
    tau_sweep : array (real)
        The list of pulse durations applied in the Ramsey-style experiment, i.e. the independent variable.
    bb_awg : array (real)
        The BB AWG pulse to be played. Before DPD, this is a step pulse. After obtaining the results, it's predistorted. 
    target_amplitude : float
        The amplitude of the BB AWG pulse that is played in the experiment. 

    Returns
    -----------
    results_x : array (real)
        The results from the experiment in the case when the final pulse played is an X-\pi/2.
    results_y : array (real)
        The results from the experiment in the case when the final pulse played is a Y-\pi/2.
    """

    T_sep = 100e-9
    start_tau = tau_sweep[0]

    results_x = np.zeros(num_samples)
    results_y = np.zeros(num_samples)

    # X measurement 
    for k in range(num_samples):
        array = np.zeros(num_samples)
        array[:k] = bb_awg[:k]
        env = qcs.ArbitraryEnvelope(tau_sweep, array)

        xmeas_sequence = copy.deepcopy(init_program)
        xmeas_sequence.n_shots(500)
        xmeas_sequence.add_waveform(pi2_ypulse, xy_awgs[1], new_layer=True)

        xmeas_sequence.add_waveform(
            pulse = qcs.DCWaveform(
                duration = pulse_duration,
                envelope = env,
                amplitude = target_amplitude
            ),
            channels = qubit_flux_awgs[0],
            new_layer = True,
            pre_delay = 10e-9
        )

        xmeas_sequence.add_waveform(pulse = pi2_xpulse, channels = xy_awgs[1], new_layer = True, pre_delay = T_sep)

        xmeas_sequence.add_waveform(readout_xpulse, channels = readout_awgs[1], new_layer = True)
        xmeas_sequence.add_acquisition(qcs.IntegrationFilter(readout_xpulse), channels = dig_channel[1], classifier = qcs.Classifier(classifier_ref), pre_delay = 53.3E-9)

        res = backend.apply(xmeas_sequence)
        results_x[k] = np.average(next(iter(res.results.get_classified(dig_channel[1]).values())))
        time.sleep(0.3)

    # Y measurement
    for k in range(num_samples):
        array = np.zeros(num_samples)
        array[:k] = bb_awg[:k]
        env = qcs.ArbitraryEnvelope(tau_sweep, array)

        ymeas_sequence = copy.deepcopy(init_program)
        ymeas_sequence.n_shots(500)
        ymeas_sequence.add_waveform(pi2_ypulse, xy_awgs[1], new_layer=True)

        ymeas_sequence.add_waveform(
            pulse=qcs.DCWaveform(
                duration = pulse_duration,
                envelope = env,
                amplitude = target_amplitude
            ),
            channels = qubit_flux_awgs[0],
            new_layer = True,
            pre_delay = 10e-9
        )

        ymeas_sequence.add_waveform(pulse = pi2_ypulse, channels = xy_awgs[1], new_layer = True, pre_delay = T_sep)

        ymeas_sequence.add_waveform(readout_ypulse, channels = readout_awgs[1], new_layer = True)
        ymeas_sequence.add_acquisition(qcs.IntegrationFilter(readout_ypulse), channels = dig_channel[1], classifier = qcs.Classifier(classifier_ref), pre_delay = 53.3E-9)

        res = backend.apply(ymeas_sequence)
        results_y[k] = np.average(next(iter(res.results.get_classified(dig_channel[1]).values())))
        time.sleep(0.3)

    ymeas_sequence.draw()

    np.save(f"XResults_Cryoscope_Method1_{start_tau*1e9:.3g}_start_{num_samples}_shots_amplitude_{target_amplitude}", results_x)
    np.save(f"YResults_Cryoscope_Method1_{start_tau*1e9:.3g}_start_{num_samples}_shots_amplitude_{target_amplitude}", results_y)

    return results_x, results_y

In [4]:
def import_method1_results():
    """
    If the experiment cannot be run for whatever reason, one can instead import the data from the last time
    that it was by calling this function. 

    Parameters
    -----------
    None. 
    
    Returns
    -----------
    results_x : array (real)
        The results from the experiment in the case when the final pulse played is an X-\pi/2.
    results_y : array (real)
        The results from the experiment in the case when the final pulse played is a Y-\pi/2.
    pulse_duration : float
        The longest BB AWG pulse to be played in the experiment.
    num_samples : int (positive)
        Sets the number of BB AWG pulses of increasing duration that should be played.
    tau_sweep : array (real)
        The list of pulse durations applied in the Ramsey-style experiment, i.e. the independent variable.
    delta_tau : float
        The spacing interval between subsequent pulses that are played in the experiment. It is the inverse of the FPGA clock.
    bb_awg : array (real)
        The BB AWG pulse to be played. Before DPD, this is a step pulse.
    target_amplitude : float
        The amplitude of the BB AWG pulse that is played in the experiment.
    """

    filename = "XResults_Cryoscope_Method1_0_start_240_shots_amplitude_0.02.npy"
    stem = Path(filename).stem

    target_amplitude = float(stem.split("amplitude_")[1])
    start_tau = float(stem.split("Method1_")[1].split("_")[0]) * 10e-10
    num_samples = int(stem.split("start_")[1].split("_")[0])
    dac_sr = 2.4e9
    tau_sweep = np.arange(num_samples) / dac_sr
    delta_tau = tau_sweep[1] - tau_sweep[0]
    bb_awg = np.heaviside(tau_sweep, 1.0) 

    results_x = np.load("XResults_Cryoscope_Method1_0_start_240_shots_amplitude_0.02.npy")
    results_y = np.load("YResults_Cryoscope_Method1_0_start_240_shots_amplitude_0.02.npy")

    return results_x, results_y, pulse_duration, num_samples, tau_sweep, delta_tau, bb_awg, target_amplitude


In [5]:
def plot_method1_results(tau_sweep, results_x, results_y):
    """
    From the results of using Method 1 of Cryoscope, the expectation value results of X and Y (after being normalised) are plotted first
    as oscillating sine and cosine waves (respectively) and then secondly together to form an (almost) unit circle.
    The results are then combined together to find the overall signal as an array of complex numbers as s = X + iY.

    Parameters
    -----------
    tau_sweep : array (real)
        The list of pulse durations applied in the Ramsey-style experiment, i.e. the independent variable.
    results_x : array (real)
        The results from the experiment in the case when the final pulse played is an X-\pi/2.
    results_y : array (real)
        The results from the experiment in the case when the final pulse played is a Y-\pi/2.

    Returns
    -----------
    s_avg : array (complex)
        The combined results from the Ramsey-style experiment as an array of complex numbers. 

    """

    min_x = np.min(results_x)
    max_x = np.max(results_x)
    min_y = np.min(results_y)
    max_y = np.max(results_y)

    norm_x = (results_x - min_x) / (max_x - min_x)
    norm_y = (results_y - min_y) / (max_y - min_y)
    

    # should see a sine wave and a cosine wave
    plt.plot(tau_sweep * 1e9, norm_x, 'o-', label = 'X Measurements')
    plt.plot(tau_sweep * 1e9, norm_y, 'o-', label = 'Y Measurements')

    plt.xlabel("Time [ns]")
    plt.ylabel("Normalized State Probability")

    plt.title("Qubit 1 Ramsey-style Experiment")
    plt.legend()
    plt.grid()
    plt.show()

    expect_x = norm_x * 2 - 1
    expect_y = norm_y * 2 - 1

    s_avg = expect_x + 1j * expect_y

    plt.figure()

    # should see a (almost) unit circle
    sc = plt.scatter(expect_x, expect_y, 
                    c=tau_sweep * 1e9, 
                    cmap='viridis',   
                    marker='o')

    plt.xlabel(r'$\langle{X}\rangle$')
    plt.ylabel(r'$\langle{Y}\rangle$')
    plt.title("2D Plot of Expectation Value Results")
    plt.grid(True)
    plt.gca().set_aspect('equal', adjustable='box')

    cbar = plt.colorbar(sc)
    cbar.set_label("Time [ns]")

    plt.show()

    return s_avg

------------------------------------

### Method 2

-----------------------------------------

This is the next method utilising the traditional way of writing an experiment using Keysight's QCS Experiment Class, limited by not being able to play pulses < 13.333ns.

In [6]:
def method2_cryoscope(start_tau, delta_tau, tau_shots, end_tau, tau_sweep, bb_awg, target_amplitude):
    """
    Executes the Ramsey-style experiment but with the BB AWG pulse being played as a single pulse of fixed duration (not swept like in Method 1). The results
    are given as the X and Y expectation values. The results are also saved as NumPy arrays so the data can be accessed
    and used without having to re-run the experiment. 

    Parameters
    -----------
    start_tau : float
        The starting duration pulse to be played.
    delta_tau : float
        The spacing interval between subsequent pulses that are played in the experiment. It is the inverse of the FPGA clock.
    tau_shots : int (positive)
        The number of BB AWG pulses played in the experiment. 
    end_tau : float
        The longest duration BB AWG pulse played in the experiment. 
    tau_sweep : array (real)
        The list of pulse durations applied in the Ramsey-style experiment, i.e. the independent variable.
    bb_awg : array (real)
        The BB AWG pulse to be played. Before DPD, this is a step pulse. After obtaining the results, it's predistorted. 
    target_amplitude : float
        The amplitude of the BB AWG pulse that is played in the experiment. 

    Returns
    -----------
    s_avg : array (complex)
        The combined results from the Ramsey-style experiment as an array of complex numbers.
    """

    t2_program = copy.deepcopy(init_program)
    tau = qcs.Scalar("tau", dtype = float)
    T_sep = qcs.Scalar("T_sep", value = 100e-9 + end_tau, dtype = float)
    t_diff = T_sep - tau

    awg_pulse = qcs.RFWaveform(
        duration = tau,
        envelope = qcs.ArbitraryEnvelope(times = tau_sweep, amplitudes = bb_awg), 
        amplitude = target_amplitude,
        rf_frequency = 0
    )

    # 1. Y_\pi/2 pulse
    t2_program.add_waveform(pi2_ypulse, xy_awgs[1])
    # 2. AWG pulse for tau long
    t2_program.add_waveform(awg_pulse, channels = qubit_flux_awgs[0], new_layer = True)
    # 3. Y_\pi/2 pulse after t_diff from step 1.
    t2_program.add_waveform(pi2_ypulse, xy_awgs[1], new_layer = True, pre_delay = 100e-9)
    # 4. Readout
    t2_program.add_waveform(readout_ypulse, readout_awgs[1], new_layer = True)
    t2_program.add_acquisition(integration_filter = readout_ypulse, channels = dig_channel[1], new_layer = False, pre_delay = 50e-9) # time of flight delay between photon arrival and acquisition start

    t2_program.draw()

    fitter = Exponential() 
    pre_processor = IQuadrature()

    cryoscope = Experiment(
        backend = backend,
        calibration_set = calibration_set,
        targets = qubits,
        program = t2_program,
        fitter = fitter,
        pre_processor = pre_processor,
        name = "Cryoscope"
    )

    n_shots = 1000

    cryoscope.configure_repetitions(
        n_shots = n_shots,
        hw_sweep = 0,
        tau = tau_sweep
    )

    cryoscope.execute(name = f"Cryoscope experiment with {n_shots} shots")

    cryoscope.plot_iq() # does it average across all of the shots? Yes, it does

    s = cryoscope.get_iq_array() # so is plot_iq plotting the mean average? A: Yes, it is
    # s: only one channel but this would differentiate between the different acquisiton channels
    # s[0]: stores the results for each of the 500 shots, i.e. each entry is s_tau1, s_tau2, ...
    # s[0][0]: stores the 500 IQ points for the first tau, i.e. each entry is s_tau1_shot1, s_tau1_shot2, ...
    s_avg = np.mean(s[0], axis=1)

    I = np.real(s_avg)
    Q = np.imag(s_avg)

    np.save(f"IData_Cryoscope_Method2_{start_tau*1e9:.3g}_start_{tau_shots}_shots_amplitude_{target_amplitude}", I)
    np.save(f"QData_Cryoscope_Method2_{start_tau*1e9:.3g}_start_{tau_shots}_shots_amplitude_{target_amplitude}", Q)

    # this is the same graph as the IQ magnitude of cryoscope.plot_iq() used to check that s_avg is correctly extracted

    plt.figure(figsize = (8,4))
    plt.plot(tau_sweep / 1e-9, np.angle(s_avg), '--') 
    plt.xlabel("Delay time [ns]")
    plt.ylabel("Acquired IQ phase [rad]")
    plt.title("Average Phase of S Data From Method 2 of Ramsey Experiment")
    plt.grid(True)
    plt.show()

    return s_avg

---------------------------------

## Data Processing for Cryoscope
----------------------------------
The following steps implement the demodulation, unwrapping of the angle, and SG smoothing filter as the 'data cleaning' steps for the Cryoscope technique. Finally, it finds the reconstructed magnetic flux.

### Fourier Transform for Demodulation

-------------------------------------

A FFT is performed on the output signal to demodulate the signal and remove the highest-magnitude frequency component.


In [ ]:
def demodulation(s_avg, delta_tau, tau_sweep, demod):
    """
    Returns the demodulated magnetic flux. The highest magnitude frequency in the data
    is found (above a certain threshold) and removed from the signal. This is to remove
    unwanted high-frequency components in the signal. 

    Parameters
    -----------
    s_avg : array (complex)
        The voltage-to-flux response obtained from the results of the Ramsey-style experiment. 
    tau_sweep : array (real)
        The list of pulse durations applied in the Ramsey-style experiment, i.e. the independent variable.
    delta_tau : float
        The spacing interval between subsequent pulses that are played in the experiment. It is the inverse of the FPGA clock.
    demod : bool
        Whether to perform demodulation or not. If False, the I and Q data are just taken as the real and imaginary parts of s_avg 
        without any demodulation. If True, the highest magnitude frequency component is removed from the signal and then the I and Q data are taken.

    Returns
    -----------
    I : array (real)
        The I data of the experiment after demodulation.
    Q : array (real)
        The Q data of the experiment after demodulation.
    f_h : float (real)
        The highest magnitude frequency found in the signal, which is removed from the signal if demodulation is performed. This is used in the formula to find the flux response.
    """

    S = fft(s_avg) 
    freqs = fftfreq(len(s_avg), d = delta_tau) # sampling interval is the interval between the pulses

    peaks, _ = find_peaks(np.abs(S), height = 1.0)
    sorted_peaks = peaks[np.argsort(np.abs(S[peaks]))]

    f_highest = freqs[sorted_peaks[-1]]
    f_second = freqs[sorted_peaks[-2]]

    print(f"Highest magnitude frequency present in the signal: {f_highest:.2e} Hz")
    print(f"Second highest magnitude frequency present in the signal: {f_second:.2e} Hz")

    # this way sets the frequency to zero and then does an IFFT
    """S_demod = S.copy()

    bandwith = 0.01e9 # 10 MHz bandwidth around the carrier frequency to remove
    mask = np.abs(freqs - f_second) < bandwith

    S_demod[mask] = 0

    s_demod = ifft(S_demod)"""

    # this way subtracts off the frequency component from the signal
    A = np.mean(s_avg * np.exp(-1j * 2 * np.pi * f_second * tau_sweep)) # demodulation to find the complex amplitude of the carrier frequency
    s_demod = s_avg - A*np.exp(1j * 2 * np.pi * f_second * tau_sweep) # remove the carrier frequency component from the signal

    S_demod = fft(s_demod) # FFT of the demodulated signal to check that the carrier frequency has been removed

    second_idx = np.argmin(np.abs(freqs - f_second))

    if demod == True:
        if np.abs(S[second_idx]) > 10.0: # if the carrier frequency is above a certain threshold, then we can be confident that it is not just noise and should be removed

            I = np.real(s_demod)
            Q = np.imag(s_demod)

        else:
            I = np.real(s_avg)
            Q = np.imag(s_avg)

    else:

        I = np.real(s_avg)
        Q = np.imag(s_avg)


    plt.figure(figsize = (8,4))
    plt.plot(freqs, np.abs(S), '-', label = 'FT of Signal Before Demodulation')
    if demod == True:
        plt.plot(freqs, np.abs(S_demod), '--', label = 'FT of Signal After Demodulation')
    plt.title("FFT of signal $s(t)$")
    plt.xlabel("Frequency (GHz)")
    plt.ylabel("Magnitude $|F\{s\}(\omega)|$")
    plt.grid(True)
    plt.axvline(f_highest, linestyle='--') # highlight f_h
    #plt.axvline(f_second, linestyle='--', color = 'red') # highlight f_second
    plt.legend()
    plt.show()



    plt.plot(tau_sweep * 1e9, I, 'o-', markersize = 2, label = 'I Data')
    plt.plot(tau_sweep * 1e9, Q, 'o-', markersize = 2, label = 'Q Data')

    plt.xlabel("Time [ns]")
    if demod == True:
        plt.ylabel("Demodulated I/Q Data")
    else:
        plt.ylabel("Raw I/Q Data")

    plt.title("Qubit 1 Ramsey-style Experiment")
    plt.legend()
    plt.grid()
    plt.show()


    return I, Q, f_highest

### Unwrapping of Accumulated Phase

---------------------------------

The accumulated phase is found by finding the angle between the I and Q data. Due to -\pi/2 < arctan(x) < +\pi/2, there will be discontinuities as the accumulated phase goes beyond this range. Thus, the angle needs to be unwrwapped to end up with a continuous function that can be differentiated.


In [ ]:
def unwrap_phase(I, Q, tau_sweep):
    """
    Returns the accumulated phase after unwrapping it to obtain a continuous function.

    Parameters
    -----------
    I : array (real)
        The I data of the experiment after demodulation.
    Q : array (real)
        The Q data of the experiment after demodulation.
    tau_sweep : array (real)
        The list of pulse durations applied in the Ramsey-style experiment, i.e. the independent variable.


    Returns
    -----------
    phi_unwrapped : array (real)
        The accumulated phase of the qubit, after unwrapping.
    """

    s_avg = I + 1j * Q

    phi_unwrapped = np.unwrap(np.angle(s_avg)) # unwrap the phase to get a continuous function
    phi_wrapped = np.angle(s_avg) # wrapped phase for comparison
    phi_unwrapped -= phi_unwrapped[0] # set the initial phase to 0

    plt.figure(figsize=(8,4))
    plt.plot(tau_sweep / 1e-9, phi_wrapped, '-', label='Demodulated wrapped phase')
    plt.plot(tau_sweep / 1e-9, phi_unwrapped, '-', label='Demodulated unwrapped phase') # continuous function
    plt.xlabel('Tau [ns]') 
    plt.ylabel('Phase [rad]')
    plt.title('Wrapped vs Unwrapped Phases (demodulated)')
    plt.legend()
    plt.grid(True)
    plt.hlines(np.pi, tau_sweep[0] / 1e-9, tau_sweep[-1] / 1e-9, linestyle='--', color='gray')
    plt.hlines(-np.pi, tau_sweep[0] / 1e-9, tau_sweep[-1] / 1e-9, linestyle='--', color='gray')
    plt.show()

    return phi_unwrapped

### SG-Filter

---------------------------------

We would also like the accumulated phase, as well as being continuous, to also be a smooth function to make it differentiable (and remove stochastic noise?). A Savitsky-Golar filter is applied for this purpose



In [ ]:
def sg_filter(tau_sweep, phi_unwrapped, delta_tau):
    """
    Returns the unwrapped phase after smoothing it out using
    an SG-filter.

    Parameters
    -----------
    phi_unwrapped : array (real)
        The accumulated phase of the qubit, after unwrapping.
    tau_sweep : array (real)
        The list of pulse durations applied in the Ramsey-style experiment, i.e. the independent variable.


    Returns
    -----------
    phi_smooth : array (real)
        The unwrapped phase of qubit after being smoothed out using an SG-filter.
    """

    window_size = 25 # HAS to be an odd number
    poly_order = 2
    phi_smooth = savgol_filter(phi_unwrapped, window_size, poly_order)
    phi_smooth = np.array(phi_smooth)

    plt.plot(tau_sweep / 1e-9, phi_unwrapped, '-', lw = 4, label='Unwrapped Phase')
    plt.plot(tau_sweep / 1e-9, phi_smooth, '-', lw = 1.5, label='Smoothed Phase')
    plt.xlabel('Tau [ns]')
    plt.ylabel('Phase [rad]')
    plt.title('Unwrapped vs Smoothed Unwrapped Phase')
    plt.legend()
    plt.grid(True)
    plt.show()

    detune_freq = savgol_filter(phi_unwrapped / (2 * np.pi), window_size, poly_order, deriv = 1) / delta_tau

    plt.plot(tau_sweep / 1e-9, detune_freq / 1e6, '-') 
    plt.grid()
    plt.xlabel("Time [ns]")
    plt.ylabel("Detuning [MHz]")
    plt.title("Qubit Frequency Detuning as a Function of Pulse Duration")
    plt.show()

    return detune_freq

### Voltage-to-Flux Response

---------------------------------

Finally, we use eq. 1 from the Cryoscope paper and invert it to reconstruct the qubit's volage-to-flux response. This is the signal that characterises the DDs that we aim to correct.


In [ ]:
def flux_response(detune_freq, tau_sweep, target_amplitude, f_h):
    """
    Returns the voltage-to-flux response of the qubit during the Ramsey-style experiment, i.e. the main result.

    Parameters
    -----------
    detune_freq : array (real)
        The detuned frequency of the qubit for each pulse played in the experiment. 
    tau_sweep : array (real)
        The list of pulse durations applied in the Ramsey-style experiment, i.e. the independent variable.
    target_amplitude : float
        The amplitude of the BB AWG pulse that is played in the experiment. 
    f_h : float (real)
        The highest magnitude frequency found in the signal, which is removed from the signal if demodulation is performed. This is used in the formula to find the flux response.


    Returns
    -----------
    signal : array (real)
        The normalised ratio of the applied voltage to the magnetic flux response. This is the end result of the Cryoscope experiment. 
    """

    Phi = np.arccos( (f_h + detune_freq + pulse_freq) ** 2 / ((wmax*1e9) ** 2) ) * (fluxquanta / np.pi) + offset 
    signal = Phi / target_amplitude 

    signal /= np.mean(signal) # normalising flux

    plt.plot(tau_sweep / 1e-9, signal, '-') 
    plt.grid()
    plt.xlabel("Time [ns]")
    plt.ylabel("Magnetic Flux Ratio (Φ/V)")
    plt.title("Qubit's Flux-Response-to-Voltage Ratio as a Function of Pulse Duration")
    #plt.xlim(0, 100)
    plt.show()

    return signal

-------------------------------

## DPD Algorithm

---------------------------------

Now we move onto the application of Cryoscope. We aim to find the coefficients of the IIR and FIR filters that can then be applied in real-time which will correct the DDs of the voltage-to-flux response. 

This section makes use of OOP and is written using functions, unlike previous sections of the notebook. This is because the in-between stages here are not interesting nor important. Thus, the user can skip to the end and specify the number of IIR filters desired (only one FIR filter will be used) and run the final cell which calls the functions.

The IIR filters assume an exponential under/over-shoot step response from the QPU which it aims to compensate for. 


### IIR Filters Calculation

------------------------

In [ ]:
def exponential_fit_multi(signal, tau_sweep, n_iir):
    """
    Calculates the amplitude and time constants needed to find the coefficients for the first-order IIR filters. 

    Parameters
    -----------
    signal : array (real)
        The normalised ratio of the applied voltage to the magnetic flux response. This is the end result of the Cryoscope experiment. 
    tau_sweep : array (real)
        The list of pulse durations applied in the Ramsey-style experiment, i.e. the independent variable.
    n_iir : int (positive)
        The number of IIR filters to be applied. 


    Returns
    -----------
    A_list : list (real)
        The list of n_iir A constants (amplitudes) for each exponential mode predicted in the output data. 
    tau_list : list (real)
        the list of n_iir tau time constants for each exponential mode predicted in the output data.
    g : float (real)
        The DC gain of the response.
    """

    target = np.ones_like(tau_sweep)

    def exp_model(t, *params):
        A = params[:n_iir]
        tau = params[n_iir:-1]
        g = params[-1]

        s = np.ones_like(t)
        for Ak, tauk in zip(A, tau):
            s += Ak * np.exp(-t / tauk)

        return target * (g * s)

    # initial guesses
    A0 = np.full(n_iir, -0.5)
    tau0 = np.logspace(
        -8,
        -2,
        n_iir
    )
    g0 = 1.0 

    p0 = np.concatenate([A0, tau0, [g0]])

    lower_bounds = np.concatenate([
    np.full(n_iir, -0.99),      # A > -1
    np.full(n_iir, 3.33e-9),    # tau > delta_t
    [0.0]                       # g > 0
    ])

    upper_bounds = np.concatenate([
    np.full(n_iir, 0.99),        # A < 1
    np.full(n_iir, 1.0),        # tau < 1s
    [np.inf]
    ])

    popt, _ = curve_fit(
        exp_model,
        tau_sweep,
        signal,
        p0=p0,
        bounds=(lower_bounds, upper_bounds),
        maxfev=2000000
    )

    A_list = popt[:n_iir]
    tau_list = popt[n_iir:-1]
    g = popt[-1]

    #print(f"List of fitted amplitude: {A_list}")
    #print(f"List of fitted time constants: {tau_list / 1e9}")
    #print(f"The gain is {g}")

    # fitted forward response (diagnostic)
    fit_full = exp_model(tau_sweep, *popt)

    plt.plot(tau_sweep*1e9, signal, label = 'Original Signal', alpha = 0.7, color = 'red')
    plt.plot(tau_sweep*1e9, fit_full, label = 'Fitted Signal', alpha = 0.7, color = 'lime')
    if n_iir > 1:
        plt.title(f'Multi-Exponential Rise Fit with {n_iir} IIR Filters')
    else:
        plt.title('Multi-Exponential Rise Fit with 1 IIR Filter')
    plt.xlabel('Time (ns)')
    plt.ylabel('Amplitude')
    plt.legend()
    plt.grid(True)
    plt.show()

    return A_list, tau_list, g

In [ ]:
def IIR_filter_calc_multi(A_list, tau_list, delta_tau):
    """
    Returns the voltage-to-flux response of the qubit during the Ramsey-style experiment, i.e. the main result.

    Parameters
    -----------
    A_list : list (real)
        The list of n_iir A constants (amplitudes) for each exponential mode predicted in the output data. 
    tau_list : list (real)
        the list of n_iir tau time constants for each exponential mode predicted in the output data.
    delta_tau : float
        The spacing interval between subsequent pulses that are played in the experiment. It is the inverse of the FPGA clock.

    Returns
    -----------
    b_tot : array (real)
        The list of feedforward coefficients (taps) for all the IIR filters specified.
    a_tot : array (real)
        The list of feedback coefficients (taps) for all the IIR filters specified.
    """

    fs = 1 / delta_tau
    b_list = []
    a_list = []

    for A, tau in zip(A_list, tau_list):

        alpha = 1 - np.exp(-1 / (fs * tau * (1 + A))) 
        #if not (0 <= alpha < 1):
        #    raise ValueError(f"Unstable alpha: {alpha}")

        if A < 0:
            k = A / ((1 + A) * (1 - alpha))
        else:
            k = A / (1 + A - alpha)

        b = np.array([
            1 - k + (k * alpha),
            -(1 - k) * (1 - alpha)
        ])

        a = np.array([
            1,
            -(1 - alpha)
        ])

        b_list.extend(b)
        a_list.extend(a)


    b_tot = np.array(b_list)
    a_tot = np.array(a_list)

    bmax = 2 - 2**-20
    highest_b = np.max(np.abs(b_tot))

    amax = 1 - 2**-20

    if highest_b > bmax:
        b_tot = 2 * b_tot / highest_b

    if np.any(np.abs(a_tot)) > amax:
        a_tot[a_tot > amax] = amax
        a_tot[a_tot < -amax] = -amax


    #print(f"The feedforward taps are: {b_tot}")
    #print(f"The feedback taps are : {a_tot}")

    return b_tot, a_tot


In [ ]:
def calc_multi_IIR_correction(feedforward_coeffs, feedback_coeffs, signal, g):
    """
    Returns the voltage-to-flux response of the qubit during the Ramsey-style experiment, i.e. the main result.

    Parameters
    -----------
    feedforward_coeffs : array (real)
        The list of feedforward coefficients (taps) for all the IIR filters specified.
    feedback_coeffs : array (real)
        The list of feedback coefficients (taps) for all the IIR filters specified.
    signal : array (real)
        The normalised ratio of the applied voltage to the magnetic flux response. This is the end result of the Cryoscope experiment. 
    g : float (real)
        The DC gain of the response.

    Returns
    -----------
    iir_correction : array (real)
        The corrected response of the signal using IIR filters. 
    """
    iir_correction = signal.copy()

    for i in range(0, len(feedback_coeffs), 2):
        feedforwards = feedforward_coeffs[i:i+2]
        feedbacks = feedback_coeffs[i:i+2]
        iir_correction = lfilter(feedforwards, feedbacks, iir_correction)
    
    iir_correction /= g  # compensate for gain
    iir_correction = np.divide(iir_correction, np.mean(iir_correction)) # normalising

    return iir_correction

### FIR Filter Calculation (w/ realistic memory)

--------------------------------

In [ ]:
def FIR_filter_calc_cutoff(signal, n_taps, tau_sweep, P): 
    """
    Calculates the feedforward taps for the FIR filter. This way calculates the coefficients only once the FIR filter
    memory is full resulting in a slow rise time that's dependent on the number of taps specified.

    This method also allows optionality for compensating for  non-linear effects through the parameter P which 
    sets the highest polynomial order of non-linearity to compensate for based on the PH PA model.

    Parameters
    -----------
    signal : array (real)
        The normalised ratio of the applied voltage to the magnetic flux response. This is the end result of the Cryoscope experiment. 
    tau_sweep : array (real)
        The list of pulse durations applied in the Ramsey-style experiment, i.e. the independent variable.
    n_taps : int (positive)
        The number of taps for the FIR filter to calculate.
    P : int (positive)
        The polynomial order of non-linearity to compensate for using the PH PA model. Can set to 1 for a regular FIR filter.

        
    Raises
    -----------
    ValueError 
        If the number of FIR taps is twice greater than the length of the signal then the FIR filter cannot
        reach full memory thus it will not work. 

    Returns
    -----------
    b : list (real)
        The list of feeforward taps for the FIR filter.
    """

    # P allows for the option of implementing the PH PA Model to account for non-linear effects

    if 2*n_taps > len(signal):
        raise ValueError(f"Number of taps is too large for the length of the signal and there cannot be a convolution matrix created. Please reduce n_taps. \n It should be <={len(signal)/2}")

    cutoff = tau_sweep[n_taps]
    mask = tau_sweep >= cutoff # this is to ensure that the X convolution matrix has no zero entries

    x = signal[mask]
    N = len(x) + 1

    # desired output is the step function
    u = np.ones(N)

    # build convolution matrix
    X = np.zeros((N, P*n_taps))

    #print(f"Shape of X is {np.shape(X)} and n_taps is {n_taps}")

    for n in range(N):
        for i in range(n_taps):
            for p in range(P):
                X[n, i+p] = (signal[n - i + (n_taps - 1)]) ** (p + 1)

    lambda_reg = 10e-4 # regularisation 
    XtX = X.T @ X


    """print("Rank:", np.linalg.matrix_rank(X))
    print("Condition number:", np.linalg.cond(X))

    print("Changed Rank:", np.linalg.matrix_rank(XtX))
    print("Changed Condition number:", np.linalg.cond(XtX))"""
    
    # least-squares solution
    #b = np.linalg.lstsq(X, u, rcond = None)[0]
    b = np.linalg.lstsq(XtX + lambda_reg * np.eye(n_taps), X.T @ u, rcond = None)[0]

    #print(f"The input signal: {signal}")
    #print(f"X Convolution Matrix: {X}")
    #print(f"FIR filter coefficients: {b}")

    return b


### FIR Filter Calculation (unphysical memory)

---------------------------------------------

In [ ]:
def FIR_filter_calc(signal, n_taps): 
    """
    Calculates the feedforward taps for the FIR filter. This way calculates the coefficients even when the FIR filter hasn't reached full
    memory. Consequently, the FIR filter output has strange, unphysical memory effects.

    Parameters
    -----------
    signal : array (real)
        The normalised ratio of the applied voltage to the magnetic flux response. This is the end result of the Cryoscope experiment. 
    n_taps : int (positive)
        The number of taps for the FIR filter to calculate.

    Returns
    -----------
    b : list (real)
        The list of feeforward taps for the FIR filter.
    """
    
    x = signal
    N = len(x)

    # desired output is the step function
    u = np.ones(N)
    # give it a realistic rise time
    #u[:3] = signal[:3]

    # build convolution matrix
    X = np.zeros((N, n_taps))

    #print(f"Shape of X is {np.shape(X)} and n_taps is {n_taps}")

    for n in range(N):
        for i in range(n_taps):
            if n - i >= 0:
                X[n, i] = x[n - i]

    # least-squares solution
    b, *_ = np.linalg.lstsq(X, u, rcond=None)

    #print(f"FIR filter coefficients: {b}")
    #print(f"X Convolution Matrix: {X}")

    return b

In [7]:
def apply_FIR_filter_taps(feedforward_taps, signal): 
    """
    Applies the feedforward taps of the FIR filter to the signal to see what the output will be.

    Parameters
    -----------
    signal : array (real)
        The normalised ratio of the applied voltage to the magnetic flux response. This is the end result of the Cryoscope experiment. 
    feedforward_taps : list (real)
        The list of feeforward taps for the FIR filter.



    Returns
    -----------
    fir_correction : array (real)
        The signal after being corrected using an FIR filter. 
    """

    fir_correction = lfilter(feedforward_taps, 1.0, signal)
    fir_correction = np.divide(fir_correction, np.mean(fir_correction))  # normalize the correction
    #iir_correction = np.divide(iir_correction, np.mean(iir_correction))  # normalize the correction
    #iir_correction = np.divide((iir_correction - np.min(iir_correction)), np.max(iir_correction) - np.min(iir_correction))  # normalize between 0 and 1
    #iir_correction = 1.0 + 0.01 * iir_correction # cheating to normalise

    return fir_correction 

### Corrected Signal Plotting

--------------------------------

In [ ]:
def plot_first_IIR_corrected_signal(signal, iir_correction, tau_sweep, n_iir):

    plt.plot(tau_sweep * 1e9, signal, label = "Original Signal", color = 'red')
    plt.plot(tau_sweep * 1e9, iir_correction, label = "IIR Corrected", color = 'blue')
    plt.xlabel("Time [ns]")
    plt.ylabel("Flux [arb. units]")
    plt.hlines(0.99, tau_sweep[0] * 1e9, tau_sweep[-1] * 1e9, colors = 'gray', linestyles = '--', label = '$\pm 1.0$%')
    plt.hlines(1.01, tau_sweep[0] * 1e9, tau_sweep[-1] * 1e9, colors = 'gray', linestyles = '--')
    plt.hlines(0.999, tau_sweep[0] * 1e9, tau_sweep[-1] * 1e9, colors = 'black', linestyles = '--', label = '$\pm 0.1$%')
    plt.hlines(1.001, tau_sweep[0] * 1e9, tau_sweep[-1] * 1e9, colors = 'black', linestyles = '--')
    plt.legend()
    #plt.ylim(0.9, 1.1)
    if n_iir > 1:
        plt.title("Corrected Flux Response Using {} IIR Filters".format(n_iir))
    else: 
        plt.title("Corrected Flux Response Using 1 IIR Filter")
    plt.grid()
    plt.show()

In [ ]:
def plot_FIR_corrected_signal(signal, fir_correction, tau_sweep, n_taps):

    plt.plot(tau_sweep * 1e9, signal, label = "Original Signal", color = 'red')
    plt.plot(tau_sweep * 1e9, fir_correction, label = "FIR Corrected", color = 'blue')
    plt.xlabel("Time [ns]")
    plt.ylabel("Flux [arb. units]")
    plt.hlines(0.99, tau_sweep[0] * 1e9, tau_sweep[-1] * 1e9, colors = 'gray', linestyles = '--', label = '$\pm 1.0$%')
    plt.hlines(1.01, tau_sweep[0] * 1e9, tau_sweep[-1] * 1e9, colors = 'gray', linestyles = '--')
    plt.hlines(0.999, tau_sweep[0] * 1e9, tau_sweep[-1] * 1e9, colors = 'black', linestyles = '--', label = '$\pm 0.1$%')
    plt.hlines(1.001, tau_sweep[0] * 1e9, tau_sweep[-1] * 1e9, colors = 'black', linestyles = '--')
    plt.legend()
    plt.title("Corrected Flux Response Using Just an FIR Filter with {} Taps".format(n_taps))
    plt.grid()
    plt.show()

In [ ]:
def plot_FIR_truncated_correction(signal, fir_correction, tau_sweep, n_taps):

    cutoff = tau_sweep[n_taps]
    mask = tau_sweep >= cutoff
    taus_cutoff = tau_sweep[mask]

    masked = fir_correction[mask]
    masked /= np.mean(masked)

    plt.plot(taus_cutoff * 1e9, signal[mask], label = "Original Signal", color = 'red')
    plt.plot(taus_cutoff * 1e9, masked, label = "FIR Corrected", color = 'blue')
    plt.hlines((1 - 5*0.001), taus_cutoff[0] * 1e9, taus_cutoff[-1] * 1e9, colors = 'gray', linestyles = '--', label = '$\pm 0.5$%')
    plt.hlines((1 + 5*0.001), taus_cutoff[0] * 1e9, taus_cutoff[-1] * 1e9, colors = 'gray', linestyles = '--')
    plt.hlines((1 - 1*0.001), taus_cutoff[0] * 1e9, taus_cutoff[-1] * 1e9, colors = 'black', linestyles = '--', label = '$\pm 0.1$%')
    plt.hlines((1 + 1*0.001), taus_cutoff[0] * 1e9, taus_cutoff[-1] * 1e9, colors = 'black', linestyles = '--')
    plt.xlabel("Time [ns]")
    plt.ylabel("Flux [arb. units]")
    plt.legend()
    plt.title("Corrected Flux Response Using Just an FIR Filter with {} Taps".format(n_taps))
    plt.suptitle("Zoomed Into the Cutoff Region")
    plt.grid()
    plt.show()

In [ ]:
def plot_corrected_signal_first_order(signal, fir_correction, tau_sweep, n_iir, n_taps):

    s_corr_norm = fir_correction #/ np.mean(fir_correction)

    plt.plot(tau_sweep * 1e9, signal, label = "Original Signal", color = 'red')
    #plt.plot(tau_sweep * 1e9, iir_correction, label = "IIR Corrected", color = 'green')
    plt.plot(tau_sweep * 1e9, s_corr_norm, label = "IIR + FIR Corrected", color = 'green')
    plt.xlabel("Time [ns]")
    plt.ylabel("Flux [arb. units]")
    plt.hlines(0.99, tau_sweep[0] * 1e9, tau_sweep[-1] * 1e9, colors = 'gray', linestyles = '--', label = '$\pm 1.0$%')
    plt.hlines(1.01, tau_sweep[0] * 1e9, tau_sweep[-1] * 1e9, colors = 'gray', linestyles = '--')
    plt.hlines(0.999, tau_sweep[0] * 1e9, tau_sweep[-1] * 1e9, colors = 'black', linestyles = '--', label = '$\pm 0.1$%')
    plt.hlines(1.001, tau_sweep[0] * 1e9, tau_sweep[-1] * 1e9, colors = 'black', linestyles = '--')
    plt.legend()
    plt.title("Corrected Flux Response Using IIR + FIR Filters")
    plt.suptitle(f"{n_iir} IIR filters and a {n_taps}-coefficient FIR filter")
    plt.grid()
    plt.show()

## Run These User Cells

--------------------------------

In [ ]:
if __name__ == "__main__":
    # --- this runs Cryoscope for the first time --- #

    # --- if using method 1, choose start_tau = 0ns, otherwise for method_2, choose start_tau > 13.3333ns --- #
    tau_shots = 30 # 60 for ~200ns end, 100 for ~330ns end, and 150 for ~500ns end
    
    method = 1
    if method == 1:
        pulse_duration = tau_shots / 300e6 # pulse durations are multiples of the FPGA clock
        dac_sr = 2.4e9 # higher resolution for cryoscope
        num_samples = int(pulse_duration * dac_sr)
        tau_sweep = np.arange(num_samples) / dac_sr
        delta_tau = tau_sweep[1] - tau_sweep[0]      

    elif method == 2:
        start_tau = 10e-9 + (1 / (300 * 1e6)) # avoid the initial fast transient
        delta_tau = 1 / (300 * 1e6) # smallest interval possible which is 1 / 300 MHz FPGA clock 
        end_tau = start_tau + tau_shots * delta_tau
        tau_sweep = np.linspace(start_tau, end_tau, tau_shots)

    target_amplitude = 0.02 # adjust this depending on the gate amplitude that you are optimising for
    bb_awg = np.heaviside(tau_sweep, 1.0) 

    if method == 1:
        results_x, results_y = method1_cryoscope(pulse_duration, num_samples, tau_sweep, bb_awg, target_amplitude)
        s_avg = plot_method1_results(tau_sweep, results_x, results_y)
    elif method == 2:
        s_avg = method2_cryoscope(start_tau, delta_tau, tau_shots, end_tau, tau_sweep, bb_awg, target_amplitude)
    # uncomment depending on whether one wants to use imported data or run the experiment 
    # tau_sweep, delta_tau, start_tau, results_x, results_y = import_method1_results()
    I, Q, f_h = demodulation(s_avg, delta_tau, tau_sweep, demod = False)
    phi_unwrapped = unwrap_phase(I, Q, tau_sweep)
    detune_freq = sg_filter(tau_sweep, phi_unwrapped, delta_tau)
    #detune_freq = detuned_freq(phi_smooth, tau_sweep, delta_tau)
    signal = flux_response(detune_freq, tau_sweep, target_amplitude, f_h)


    n_iir = 1 # this is the number of IIR filters to be used
    A_list, tau_list, g = exponential_fit_multi(signal, tau_sweep, n_iir)
    b_tot, a_tot = IIR_filter_calc_multi(A_list, tau_list, delta_tau)
    iir_correction = calc_multi_IIR_correction(b_tot, a_tot, signal, g)
    plot_first_IIR_corrected_signal(signal, iir_correction, tau_sweep, n_iir)
    n_taps = 48 # set the number of FIR taps desired
    b = FIR_filter_calc_cutoff(signal, n_taps, tau_sweep, P=1)
    fir_correction = apply_FIR_filter_taps(b, signal)
    plot_FIR_corrected_signal(signal, fir_correction, tau_sweep, n_taps)
    plot_FIR_truncated_correction(signal, fir_correction, tau_sweep, n_taps)
    """n_taps = 48 # set the number of FIR taps desired
    b = FIR_filter_calc(signal, n_taps)
    fir_correction = apply_FIR_filter_taps(b, signal)
    plot_FIR_corrected_signal(signal, fir_correction, tau_sweep, n_taps)"""
    
    

In [ ]:
# checking what the BB AWG looks like after DPD
bb_awg = np.heaviside(tau_sweep, 1.0) #* target_amplitude

iir_filtered = lfilter(b_tot, a_tot, bb_awg)
fir_filtered = lfilter(b, 1.0, bb_awg)
#fir_filtered = lfilter(b_tot, a_tot, bb_awg)

plt.plot(tau_sweep * 1e9, bb_awg, label = 'BB AWG Pulse')
plt.plot(tau_sweep * 1e9, iir_filtered, label = 'IIR Corrected')
#plt.plot(tau_sweep * 1e9, fir_filtered, label = 'FIR Corrected')
plt.xlabel('Time [ns]')
plt.ylabel('Amplitude [a.u.]')
plt.grid(True)
#plt.xlim(25, 200)
#plt.ylim(target_amplitude * 0.999, target_amplitude * 1.001)
plt.legend()
plt.show()

In [ ]:
if __name__ == "__main__":
    # --- this is for running Cryoscope again but with the BB AWG predistorted, hopefully getting better results --- 
    bb_awg = np.heaviside(tau_sweep, 1.0)
    iir_filtered = lfilter(b_tot, a_tot, bb_awg)
    #fir_filtered = lfilter(b, 1.0, bb_awg)

    plt.plot(tau_sweep * 1e9, signal, label = 'Output Signal')
    plt.plot(tau_sweep * 1e9, iir_filtered, label = 'IIR Corrected BB AWG')
    plt.xlabel('Time [ns]')
    plt.ylabel('Amplitude [a.u.]')
    plt.grid(True)
    plt.legend()
    #plt.xlim(80, 200)
    #plt.ylim(target_amplitude * 0.99, target_amplitude * 1.01)
    plt.show()

    """if method == 1:
        results_x, results_y = method1_cryoscope(pulse_duration, num_samples, tau_shots, tau_sweep, iir_filtered, target_amplitude)
        s_avg2 = plot_method1_results(tau_sweep, results_x, results_y)
    elif method == 2:
        s_avg2 = method2_cryoscope(start_tau, delta_tau, tau_shots, end_tau, tau_sweep, iir_filtered, target_amplitude)"""
    I, Q, f_h = demodulation(s_avg2, delta_tau, tau_sweep, demod = False)

    phi_unwrapped = unwrap_phase(I, Q, tau_sweep)
    detune_freq = sg_filter(tau_sweep, phi_unwrapped, delta_tau)
    #detune_freq = detuned_freq(phi_smooth, tau_sweep2, delta_tau)
    signal2 = flux_response(detune_freq, tau_sweep, target_amplitude, f_h)